In [1]:
!pip install mitreattack-python transformers torch chromadb tqdm numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.8/556.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00

In [17]:
from mitreattack.stix20 import MitreAttackData
from transformers import AutoTokenizer, AutoModel
import torch
import chromadb
from chromadb.config import Settings
import json
import os
import numpy as np
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
from sentence_transformers import SentenceTransformer


In [9]:
!wget -q https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json -O enterprise-attack.json


In [4]:
OUT_DIR = Path(".")  # Current directory for outputs
CHROMA_DIR = Path("chroma_db")
CHROMA_DIR.mkdir(exist_ok=True)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

Using device: cuda
   GPU: Tesla T4
   Memory: 15.8GB


In [10]:
print("\nLoading MITRE ATT&CK data...")
try:
    mad = MitreAttackData(stix_filepath="enterprise-attack.json")
    print("ATT&CK data loaded successfully")
except Exception as e:
    print(f"Error loading ATT&CK data: {e}")
    print("Trying alternative approach...")
    # Fallback approach if direct loading fails
    mad = MitreAttackData("enterprise-attack")


Loading MITRE ATT&CK data...
ATT&CK data loaded successfully


In [12]:
techniques = mad.get_techniques()  # returns dict keyed by technique id-ish objects
print(f"Found {len(techniques)} techniques")

Found 823 techniques


In [13]:
print("\nProcessing ATT&CK techniques...")
docs = []
processed_count = 0

for t in tqdm(techniques, desc="Processing techniques"):
    try:
        # mitre objects expose id, name, description, tactic(s), platform, etc.
        external_refs = t.get('external_references', [])
        tid = None
        for ref in external_refs:
            if ref.get('source_name') == 'mitre-attack':
                tid = ref.get('external_id')
                break

        if not tid:
            tid = t.get('id', f'unknown_{processed_count}')

        name = t.get('name', 'Unknown Technique')
        desc = t.get('description', '') or t.get('x_mitre_shortdescription', '') or 'No description available'

        # Get tactics - handle different data structure possibilities
        tactics = []
        try:
            tactics = mad.get_tactics_for_technique(tid) or []
        except:
            # Fallback: extract from kill chain phases
            kill_chain_phases = t.get('kill_chain_phases', [])
            tactics = [phase.get('phase_name', '') for phase in kill_chain_phases if phase.get('kill_chain_name') == 'mitre-attack']

        # gather mitigations - handle API availability gracefully
        mitigation_text = ""
        try:
            if hasattr(mad, 'search_mitigations_for_technique'):
                mitigations = mad.search_mitigations_for_technique(tid)
                mitigation_text = "\n".join([
                    f"{m.get('name', 'Unknown')}: {m.get('description', 'No description')}"
                    for m in mitigations[:3]  # Limit to first 3 mitigations
                ])
        except Exception as e:
            mitigation_text = "Mitigation information not available"

        # Get platforms
        platforms = t.get('x_mitre_platforms', []) or []

        # compose a canonical text
        tactics_text = ', '.join(tactics) if tactics else 'Unknown'
        full_text = f"""Technique: {tid} — {name}
Tactic(s): {tactics_text}
Platforms: {', '.join(platforms) if platforms else 'Not specified'}

Description:
{desc}

Mitigations:
{mitigation_text if mitigation_text else 'No specific mitigations available'}"""

        doc = {
            "attack_id": tid,
            "name": name,
            "text": full_text,
            "tactics": tactics,
            "platforms": platforms,
            "raw": {k: v for k, v in t.items() if k not in ['raw']}  # Avoid deep nesting
        }
        docs.append(doc)
        processed_count += 1

    except Exception as e:
        print(f"⚠️ Error processing technique {processed_count}: {e}")
        continue

print(f"Successfully processed {len(docs)} ATT&CK techniques")


Processing ATT&CK techniques...


Processing techniques: 100%|██████████| 823/823 [00:00<00:00, 61913.27it/s]

Successfully processed 823 ATT&CK techniques


In [14]:
def chunk_text(text, max_chars=1200):  # Slightly smaller for better GPU memory management
    """Split text into chunks, preserving paragraph structure when possible"""
    if len(text) <= max_chars:
        return [text]

    paras = text.split("\n\n")
    chunks = []
    cur = ""

    for p in paras:
        if len(cur) + len(p) + 2 <= max_chars:
            cur += ("\n\n" + p) if cur else p
        else:
            if cur:
                chunks.append(cur.strip())
            # If paragraph itself is too long, split it
            if len(p) > max_chars:
                sentences = p.split('. ')
                temp_chunk = ""
                for sent in sentences:
                    if len(temp_chunk) + len(sent) + 2 <= max_chars:
                        temp_chunk += (". " + sent) if temp_chunk else sent
                    else:
                        if temp_chunk:
                            chunks.append(temp_chunk.strip())
                        temp_chunk = sent
                if temp_chunk:
                    chunks.append(temp_chunk.strip())
                cur = ""
            else:
                cur = p

    if cur:
        chunks.append(cur.strip())

    return [c for c in chunks if c.strip()]  # Remove empty chunks

print("\nChunking documents...")
chunked_docs = []
total_chunks = 0

for doc in tqdm(docs, desc="Chunking"):
    chunks = chunk_text(doc['text'])
    for i, c in enumerate(chunks):
        chunked_docs.append({
            "attack_id": doc['attack_id'],
            "name": doc['name'],
            "chunk_id": f"{doc['attack_id']}_chunk_{i}",
            "text": c,
            "metadata": {
                "tactics": doc['tactics'],
                "platforms": doc['platforms'],
                "chunk_index": i,
                "total_chunks": len(chunks)
            }
        })
    total_chunks += len(chunks)

print(f"Created {len(chunked_docs)} chunks (avg {total_chunks/len(docs):.1f} chunks per technique)")



Chunking documents...


Chunking: 100%|██████████| 823/823 [00:00<00:00, 84736.53it/s]

Created 1578 chunks (avg 1.9 chunks per technique)


In [18]:
print("\nLoading MiniLM model (all-MiniLM-L6-v2)...")
model_name = "sentence-transformers/all-MiniLM-L6-v2"

try:
    model = SentenceTransformer(model_name)
    print(f"Model loaded successfully: {model_name}")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

def get_embeddings(texts, batch_size=8):
    """Generate embeddings using MiniLM with batching"""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i+batch_size]
        batch_embeddings = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)


Loading MiniLM model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully: sentence-transformers/all-MiniLM-L6-v2


In [19]:
print("\nGenerating embeddings with ATT&CK-BERT...")
texts = [doc['text'] for doc in chunked_docs]
print(f"Processing {len(texts)} text chunks...")

try:
    embeddings = get_embeddings(texts)
    print(f"Generated embeddings shape: {embeddings.shape}")
except Exception as e:
    print(f"Error generating embeddings: {e}")
    raise

# Clear model from GPU memory
if torch.cuda.is_available():
    del model
    torch.cuda.empty_cache()
    print("Cleared model from GPU memory")



Generating embeddings with ATT&CK-BERT...
Processing 1578 text chunks...


Generating embeddings: 100%|██████████| 198/198 [00:05<00:00, 35.53it/s]

Generated embeddings shape: (1578, 384)
Cleared model from GPU memory


In [20]:
print("\nSetting up ChromaDB...")
try:
    # Use in-memory settings that work well in Colab
    chroma_client = chromadb.PersistentClient(
        path=str(CHROMA_DIR),
        settings=Settings(
            anonymized_telemetry=False,
            allow_reset=True,
            is_persistent=True
        )
    )
    print("ChromaDB client initialized")
except Exception as e:
    print(f"ChromaDB initialization error: {e}")
    raise



Setting up ChromaDB...
ChromaDB client initialized


In [21]:
collection_name = "attack_techniques"
try:
    # Try to delete existing collection if it exists
    existing_collections = [c.name for c in chroma_client.list_collections()]
    if collection_name in existing_collections:
        chroma_client.delete_collection(collection_name)
        print(f"🗑️ Deleted existing collection: {collection_name}")
except Exception as e:
    print(f"ℹ️ No existing collection to delete: {e}")

try:
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"description": "MITRE ATT&CK techniques embedded with ATT&CK-BERT"}
    )
    print(f"✅ Created collection: {collection_name}")
except Exception as e:
    print(f"❌ Error creating collection: {e}")
    raise


✅ Created collection: attack_techniques


In [22]:
print("\n📥 Adding documents to ChromaDB...")
batch_size = 50  # Conservative batch size for ChromaDB in Colab

try:
    for i in tqdm(range(0, len(chunked_docs), batch_size), desc="Adding to ChromaDB"):
        batch_docs = chunked_docs[i:i+batch_size]
        batch_embeddings = embeddings[i:i+batch_size]

        # Prepare data for ChromaDB
        ids = [doc['chunk_id'] for doc in batch_docs]
        documents = [doc['text'] for doc in batch_docs]
        metadatas = []

        for doc in batch_docs:
            # ChromaDB metadata must be strings, numbers, or booleans
            metadata = {
                'attack_id': str(doc['attack_id']),
                'name': str(doc['name']),
                'chunk_index': int(doc['metadata']['chunk_index']),
                'total_chunks': int(doc['metadata']['total_chunks']),
                'tactics': ', '.join(doc['metadata']['tactics']) if doc['metadata']['tactics'] else 'Unknown',
                'platforms': ', '.join(doc['metadata']['platforms']) if doc['metadata']['platforms'] else 'Not specified'
            }
            metadatas.append(metadata)

        # Add to collection
        collection.add(
            embeddings=batch_embeddings.tolist(),
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )

    print(f"✅ Added {len(chunked_docs)} documents to ChromaDB collection")
except Exception as e:
    print(f"❌ Error adding to ChromaDB: {e}")
    raise



📥 Adding documents to ChromaDB...


Adding to ChromaDB: 100%|██████████| 32/32 [00:03<00:00,  8.93it/s]

✅ Added 1578 documents to ChromaDB collection


In [23]:
print("\n💾 Saving chunks to JSONL...")
try:
    with open(OUT_DIR / "attack_chunks.jsonl", "w", encoding='utf-8') as f:
        for doc in chunked_docs:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    print(f"✅ Saved {len(chunked_docs)} chunks to attack_chunks.jsonl")
except Exception as e:
    print(f"❌ Error saving JSONL: {e}")
    raise

# 9) Verify ChromaDB collection
try:
    collection_count = collection.count()
    print(f"✅ ChromaDB collection '{collection_name}' contains {collection_count} documents")
except Exception as e:
    print(f"⚠️ Could not verify collection count: {e}")
    collection_count = len(chunked_docs)



💾 Saving chunks to JSONL...
✅ Saved 1578 chunks to attack_chunks.jsonl
✅ ChromaDB collection 'attack_techniques' contains 1578 documents


In [26]:
print("\n" + "="*60)
print("PROCESSING COMPLETE!")
print("="*60)
print(f"Summary:")
print(f"   • ATT&CK techniques processed: {len(docs)}")
print(f"   • Text chunks created: {len(chunked_docs)}")
print(f"   • Embeddings generated: {embeddings.shape}")
print(f"   • ChromaDB documents: {collection_count}")
print(f"\nOutput files:")
print(f"   • attack_chunks.jsonl ({os.path.getsize('attack_chunks.jsonl') / 1024 / 1024:.1f} MB)")
print(f"   • chroma_db/ directory (persistent ChromaDB store)")



PROCESSING COMPLETE!
Summary:
   • ATT&CK techniques processed: 823
   • Text chunks created: 1578
   • Embeddings generated: (1578, 384)
   • ChromaDB documents: 1578

Output files:
   • attack_chunks.jsonl (1.6 MB)
   • chroma_db/ directory (persistent ChromaDB store)


In [27]:
print("\nTesting ChromaDB query functionality...")
try:
    test_results = collection.query(
        query_texts=["lateral movement techniques"],
        n_results=3
    )
    print(f"Test query successful - returned {len(test_results['documents'][0])} results:")
    for i, doc in enumerate(test_results['documents'][0]):
        metadata = test_results['metadatas'][0][i]
        print(f"   {i+1}. {metadata['attack_id']} - {metadata['name']}")
except Exception as e:
    print(f"Test query failed: {e}")



Testing ChromaDB query functionality...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.4MiB/s]


Test query successful - returned 3 results:
   1. T1210 - Exploitation of Remote Services
   2. T1077 - Windows Admin Shares
   3. T1205 - Traffic Signaling


In [29]:
print(f"\nFile sizes for download:")
try:
    jsonl_size = os.path.getsize('attack_chunks.jsonl') / 1024 / 1024
    print(f"   • attack_chunks.jsonl: {jsonl_size:.1f} MB")

    chroma_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                     for dirpath, dirnames, filenames in os.walk('chroma_db')
                     for filename in filenames) / 1024 / 1024
    print(f"   • chroma_db/: {chroma_size:.1f} MB")
    print(f"   • Total download size: {jsonl_size + chroma_size:.1f} MB")
except:
    print("   • File size calculation unavailable")



File sizes for download:
   • attack_chunks.jsonl: 1.6 MB
   • chroma_db/: 15.0 MB
   • Total download size: 16.6 MB
